# Pandas 결측치(Missing Values) 처리

## 📚 결측치 용어 정리 (Terminology)

### 결측치 기본 용어

| 용어 (한글) | 용어 (영문) | 정의 | 표시 |
|------------|-----------|------|------|
| **결측치** | Missing Value | 데이터가 없는 상태 | `NaN`, `NA`, `None` |
| **누락값** | Null Value | 값이 존재하지 않음 | 데이터베이스 용어 |
| **공백** | Blank / Empty | 비어있는 문자열 | (빈 문자열) |
| **무한대** | Infinity | 수학적 무한 | `inf`, `-inf` |

### 결측치 유형별 상세 설명

```
데이터 타입별 결측치 표현

실수(float):  [1.0] [NaN] [3.0]  → np.nan (Not a Number)
               ↓    ↓     ↓
             유효  결측  유효

정수(Int64):  [1]   [<NA>]  [3]  → pd.NA (Not Available)
               ↓      ↓      ↓
             유효   결측   유효

문자열:      ['A']  [<NA>]  ['C'] → pd.NA
              ↓       ↓      ↓
            유효    결측   유효

날짜:        [2024-01-01] [NaT] [2024-01-03] → pd.NaT (Not a Time)
                 ↓          ↓        ↓
               유효       결측     유효
```

### 결측치 처리 방법 용어

| 용어 (한글) | 용어 (영문) | 정의 | 예시 |
|------------|-----------|------|------|
| **제거** | Deletion / Drop | 결측치가 있는 행/열 삭제 | `df.dropna()` |
| **대체** | Imputation / Fill | 결측치를 다른 값으로 채움 | `df.fillna(0)` |
| **보간** | Interpolation | 주변 값을 기준으로 추정 | `df.interpolate()` |
| **전진 채움** | Forward Fill | 이전 값으로 채움 | `df.fillna(method='ffill')` |
| **후진 채움** | Backward Fill | 다음 값으로 채움 | `df.fillna(method='bfill')` |

### 결측치 처리 전략 비교

| 전략 | 장점 | 단점 | 사용 상황 |
|------|------|------|-----------|
| **제거** | 간단함, 왜곡 없음 | 데이터 손실 | 결측치가 적을 때 (< 5%) |
| **평균 대체** | 쉬움, 분포 유지 | 분산 감소 | 수치형 데이터, 정규분포 |
| **중앙값 대체** | 이상치 영향 적음 | 쉬움 | 수치형 데이터, 비대칭 분포 |
| **최빈값 대체** | 범주형에 적합 | 정보 손실 | 범주형 데이터 |
| **보간** | 시간적 패턴 반영 | 추정 오류 | 시계열 데이터 |

### pandas에서의 결측치 종류

| 종류 | 풀네임 | 설명 | 사용 상황 | 시각적 표시 |
|------|--------|------|-----------|-------------|
| `np.nan` | Not a Number | NumPy의 실수형 결측치 | float64 컬럼 | `NaN` |
| `pd.NA` | Not Available | pandas의 새로운 결측치 | Int64, string, boolean | `<NA>` |
| `pd.NaT` | Not a Time | 날짜/시간 결측치 | datetime 컬럼 | `NaT` |
| `None` | None | Python 객체 결측치 | object 타입 | `None` |

### 중요한 차이점
- `np.nan`: float64 타입에서 사용, 연산 시 NaN 전파 (1 + NaN = NaN)
- `pd.NA`: nullable 타입(Int64, string, boolean)에서 사용, 타입 보존
- `pd.NaT`: datetime 타입에서 사용, 날짜 연산 가능

---

## 1. 결측치란?

**결측치(Missing Value)**는 데이터셋에서 값이 없는 것을 의미합니다.


In [2]:
import pandas as pd
import numpy as np

In [3]:
s=pd.Series(["Sam",np.nan,"Tim","Kim"])
s

0    Sam
1    NaN
2    Tim
3    Kim
dtype: str

In [4]:
# isnull(): 각 값이 결측치인지 확인
# 결과가 True면 해당 위치가 결측치(NaN), False면 값이 있음

s.isnull()

0    False
1     True
2    False
3    False
dtype: bool

In [5]:
# notnull(): 각 값이 결측치가 아닌지 확인 (isnull()의 반대)
# 결과가 True면 값이 있음, False면 결측치

s.notnull()

0     True
1    False
2     True
3     True
dtype: bool

In [6]:
s[3]=None
s.isnull()

0    False
1     True
2    False
3     True
dtype: bool

In [7]:
# dropna(): 결측치가 있는 행(또는 열)을 제거
# 기본적으로 결측치가 하나라도 있으면 해당 행 전체를 제거
# inplace=True 옵션으로 원본을 직접 수정할 수 있음

s.dropna()

0    Sam
2    Tim
dtype: str

In [8]:
s

0    Sam
1    NaN
2    Tim
3    NaN
dtype: str

In [9]:
s.dropna(inplace=True)

In [10]:
s

0    Sam
2    Tim
dtype: str

In [11]:
# pd.NA: pandas의 nullable 타입(Int64, string, boolean)에서 사용되는 결측치 표현
# pd.NaT: 날짜/시간(datetime) 타입에서 사용되는 결측치 표현 (Not a Time)

print(pd.NA)
print(pd.NaT)

<NA>
NaT


In [12]:
# 예제 1: np.nan을 포함하는 DataFrame
# float 타입 컬럼에서 np.nan 사용

df1 = pd.DataFrame([[1, 2, 3], [4, np.nan, 5], [np.nan, np.nan, np.nan]])


In [13]:
df1.dtypes

0    float64
1    float64
2    float64
dtype: object

In [14]:
# np.nan은 숫자이기 때문에 float 타입의 컬럼에서 그대로 있음
df1

,0,1,2
0,1.0,2.0,3.0
1,4.0,NaN,5.0
2,NaN,NaN,NaN


In [15]:
# 예제 2: None을 포함하는 DataFrame
# None은 파이썬의 NoneType이지만, float64 컬럼에서는 np.nan으로 자동 변환됨

df2 = pd.DataFrame([[1.0, 2.0], [None, 4.0]])

In [16]:
df2.dtypes

0    float64
1    float64
dtype: object

In [17]:
# None은 숫자가 아니기 때문에 float 타입의 컬럼에서 np.nan으로 변환
df2

,0,1
0,1.0,2.0
1,NaN,4.0


In [18]:
# 예제 3: 일반 int64 vs nullable Int64

# df3: 일반 리스트로 생성 → None이 np.nan으로 변환되면서 float64로 변경
# df4: dtype='Int64'로 명시 → nullable integer 타입 유지, None은 pd.NA로 변환

df3 = pd.DataFrame({"a": [1, None, 3]})
df4 = pd.DataFrame({"a": pd.Series([1, None, 3], dtype="Int64")})

In [19]:
df3

,a
0,1.0
1,NaN
2,3.0


In [20]:
df4

,a
0,1
1,<NA>
2,3


In [21]:
# 4
# NA를 사용한 이유

df5 = pd.DataFrame({"a": [1, None, 3]}) 
# 값 [1, None, 3]은 None을 np.nan으로 바꾸기만 하면 float64이 되네?
# None → np.nan이 되면서 원래는 정수였던 1, 3까지도 실수 타입으로 바뀌어 버림

print(df5.dtypes)

a    float64
dtype: object


In [22]:
df6 = pd.DataFrame({"a": pd.array([1, None, 3], dtype="Int64")})
# 값 1, 3은 정수라고 명시적으로 알림
# 이 경우 None은 np.nan이 아닌 pd.NA로 변환

In [23]:
df6

,a
0,1
1,<NA>
2,3


In [24]:
# 정리하면
# 결측치는 세 종류
# 1. 실수(float) → np.nan
# 2. 정수(Int64) → pd.NA
# 3. 날짜/시간 → pd.NaT

# 문자열의 경우도 pd.NA

df = pd.DataFrame({"a": pd.array(["hello", None, "world"], dtype="string")})
df

,a
0,hello
1,<NA>
2,world


## 2. 결측치 제거 (dropna)

### 📚 dropna 용어 및 파라미터 상세 설명

```
axis (축) 방향 이해하기

axis=0 (행 방향 - 위에서 아래로)
     ↓
   ┌───┬───┬───┐
   │ 1 │ 2 │ 3 │ ← 행 0
   ├───┼───┼───┤
   │ 4 │NaN│ 5 │ ← 행 1 (결측치 있음) → 삭제 대상
   ├───┼───┼───┤
   │NaN│NaN│NaN│ ← 행 2 (모두 결측) → 삭제 대상
   └───┴───┴───┘
   
axis=1 (열 방향 - 왼쪽에서 오른쪽으로)
     →
   ┌───┬───┬───┐
   │ 1 │ 2 │ 3 │
   ├───┼───┼───┤
   │ 4 │NaN│ 5 │
   ├───┼───┼───┤
   │NaN│NaN│NaN│
   └───┴───┴───┘
     ↑   ↑
    삭제 삭제
```

### how 파라미터 상세 설명

| 값 | 의미 | 동작 | 사용 상황 |
|----|------|------|-----------|
| `'any'` | any (어느 하나라도) | 하나라도 결측치면 제거 | 엄격한 결측치 처리 |
| `'all'` | all (모두) | 모두 결측치일 때만 제거 | 완전 빈 행/열 제거 |

```
how='any' vs how='all' 비교

원본 DataFrame:        how='any' 결과:        how='all' 결과:
┌───┬───┬───┐         ┌───┬───┬───┐         ┌───┬───┬───┐
│ 1 │ 2 │ 3 │         │ 1 │ 2 │ 3 │         │ 1 │ 2 │ 3 │
├───┼───┼───┤         └───┴───┴───┘         ├───┼───┼───┤
│ 4 │NaN│ 5 │  →                               │ 4 │NaN│ 5 │
├───┼───┼───┤                                   └───┴───┴───┘
│NaN│NaN│NaN│                                   (2행만 삭제)
└───┴───┴───┘
(1행만 삭제 -           (2행만 삭제 - 
 결측치 1개)            모두 결측치)
```

### thresh 파라미터 이해하기

```
thresh=2: 유효한 값이 2개 이상이어야 유지

원본:                  thresh=2 결과:
┌───┬───┬───┐         ┌───┬───┬───┐
│ 1 │ 2 │NaN│         │ 1 │ 2 │NaN│ ← 유효값 2개, 유지
├───┼───┼───┤         ├───┼───┼───┤
│ 4 │NaN│NaN│  →      │ 4 │NaN│NaN│? → 유효값 1개, 삭제
├───┼───┼───┤         └───┴───┴───┘
│NaN│NaN│NaN│
└───┴───┴───┘
```

### dropna() 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|----------|------|--------|
| `axis` | 0(행) 또는 1(열) 방향 | 0 |
| `how` | 'any'(하나라도) 또는 'all'(모두) | 'any' |
| `thresh` | 최소 유효 값 개수 | None |
| `subset` | 검사할 특정 컬럼 | None |
| `inplace` | 원본 수정 여부 | False |

### 예시
```python
df.dropna()                    # 결측치가 있는 행 모두 제거
df.dropna(how='all')           # 행의 모든 값이 결측치일 때만 제거
df.dropna(thresh=2)            # 유효 값이 2개 미만인 행 제거
df.dropna(subset=['A'])        # A 컬럼 기준으로 결측치 행 제거
df.dropna(axis=1)              # 결측치가 있는 열 제거
```

In [25]:
# 예제 DataFrame 생성
# 행 0: 결측치 없음
# 행 1: 1개의 결측치
# 행 2: 모든 값이 결측치

df=pd.DataFrame([[1,2,3],[4, np.nan, 5],
                 [np.nan, np.nan, np.nan]])
df

,0,1,2
0,1.0,2.0,3.0
1,4.0,NaN,5.0
2,NaN,NaN,NaN


In [26]:
df.dtypes

0    float64
1    float64
2    float64
dtype: object

In [27]:
df.dropna() # ---------- 대상은 np.nan, pd.NA, pd.NaT

,0,1,2
0,1.0,2.0,3.0


In [28]:
df.dropna(how="all")
# how="any" (디폴트) → 행에 하나라도 결측치가 있으면 제거
# how="all" → 행의 모든 값이 결측치일 때만 제거

,0,1,2
0,1.0,2.0,3.0
1,4.0,NaN,5.0


In [29]:
df.fillna(0,inplace=True)
df

,0,1,2
0,1.0,2.0,3.0
1,4.0,0.0,5.0
2,0.0,0.0,0.0


## 3. 결측치 채우기 (fillna)

### 📚 fillna 용어 및 개념 상세 설명

```
대체(Imputation) 방향 이해하기

ffill (Forward Fill) - 전진 채움
→ → → 방향 (위에서 아래로)

  행 0:   [1.0]  [2.0]  [3.0]
  행 1:   [4.0]  [NaN]  [5.0]   ← NaN을 위의 값으로 채움
                  ↓
                [2.0]   ← 행 0의 값을 복사
  행 2:   [NaN]  [NaN]  [NaN]   ← NaN을 위의 값으로 채움
            ↓      ↓
          [4.0]  [2.0]   ← 행 1의 값을 복사

결과:    [1.0]  [2.0]  [3.0]
         [4.0]  [2.0]  [5.0]
         [4.0]  [2.0]  [5.0]

bfill (Backward Fill) - 후진 채움
← ← ← 방향 (아래에서 위로)

  행 0:   [1.0]  [NaN]  [3.0]   ← NaN을 아래의 값으로 채움
                  ↓
                [2.0]   ← 행 1의 값을 복사
  행 1:   [4.0]  [2.0]  [5.0]
  행 2:   [NaN]  [2.0]  [NaN]   ← NaN을 아래의 값으로 채움 (없으면 유지)

결과:    [1.0]  [2.0]  [3.0]
         [4.0]  [2.0]  [5.0]
         [NaN]  [2.0]  [NaN]
```

### 대체 방법별 특징

| 방법 | 장점 | 단점 | 사용 상황 |
|------|------|------|-----------|
| **상수 대체** | 단순함 | 데이터 패턴 무시 | 모든 결측치를 동일값으로 채울 때 |
| **ffill** | 시간적 연속성 반영 | 첫 값이 NaN이면 채워지지 않음 | 시계열 데이터 (이전 값 유효) |
| **bfill** | 미래 값 활용 | 마지막 값이 NaN이면 채워지지 않음 | 역순 데이터 처리 |
| **평균 대체** | 전체 분포 반영 | 분산 감소, 이상치 영향 | 정규분포 데이터 |
| **중앙값 대체** | 이상치 강건 | 정확한 중심값 | 비대칭 분포 데이터 |

### fillna() 주요 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `value` | 채울 값 | 0, 'Unknown', df.mean() |
| `method` | 채우기 방법 | 'ffill'(앞값), 'bfill'(뒷값) |
| `axis` | 채우기 방향 | 0(행), 1(열) |
| `inplace` | 원본 수정 여부 | True, False |

In [30]:
df.fillna(0)              # 모든 결측치를 0으로 채우기
df.fillna(method='ffill') # 앞의 값으로 채우기 (forward fill)
df.fillna(method='bfill') # 뒤의 값으로 채우기 (backward fill)
df.fillna(df.mean())      # 평균값으로 채우기

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [31]:
# pd.NA를 포함하는 DataFrame 생성

df=pd.DataFrame([[1, 2, 3],[4, pd.NA, 5],
                 [pd.NA, pd.NA, pd.NA]])
df

,0,1,2
0,1,2,3
1,4,<NA>,5
2,<NA>,<NA>,<NA>


In [32]:
# fillna(0): 모든 결측치를 0으로 채우기
# 원본 DataFrame은 변경되지 않음 (inplace=False가 기본값)

df.fillna(0)

,0,1,2
0,1,2,3
1,4,0,5
2,0,0,0


In [33]:
df = pd.DataFrame({
    "실수": [1.0, np.nan, 3.0],
    "정수": pd.array([1, None, 3], dtype="Int64"),
    "날짜": pd.to_datetime(["2024-01-01", None, "2024-01-03"]),
    "문자": pd.array(["hello", None, "world"], dtype="string"),
})

In [34]:
df

,실수,정수,날짜,문자
0,1.0,1,2024-01-01,hello
1,NaN,<NA>,NaT,<NA>
2,3.0,3,2024-01-03,world


In [35]:
df.dropna()

,실수,정수,날짜,문자
0,1.0,1,2024-01-01,hello
2,3.0,3,2024-01-03,world


In [36]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "a": [1.0, np.nan, np.nan, 4.0],
    "b": [2.0, np.nan, 3.0,   np.nan],
    "c": [3.0, np.nan, 4.0,   5.0],
})
print(df)


     a    b    c
0  1.0  2.0  3.0
1  NaN  NaN  NaN
2  NaN  3.0  4.0
3  4.0  NaN  5.0


In [37]:

# 1. how="any" (디폴트) → 하나라도 결측치면 제거
df.dropna(how="any")


,a,b,c
0,1.0,2.0,3.0


In [38]:

# 2. how="all" → 모두 결측치일 때만 제거
df.dropna(how="all")


,a,b,c
0,1.0,2.0,3.0
2,NaN,3.0,4.0
3,4.0,NaN,5.0


In [39]:
# 3. thresh=2 → 결측치가 아닌 값이 2개 이상인 행만 유지
df.dropna(thresh=2)


,a,b,c
0,1.0,2.0,3.0
2,NaN,3.0,4.0
3,4.0,NaN,5.0


In [40]:
# 4. subset → 특정 컬럼만 기준으로 삼기
# a 컬럼 기준, a가 NaN인 행만 제거
df.dropna(subset=["a"])


,a,b,c
0,1.0,2.0,3.0
3,4.0,NaN,5.0


In [41]:
df

,a,b,c
0,1.0,2.0,3.0
1,NaN,NaN,NaN
2,NaN,3.0,4.0
3,4.0,NaN,5.0


In [42]:
# axis=1 → 행이 아닌 열(column) 기준으로 제거
# how="any" → 하나라도 결측치가 있는 열을 모두 제거
# 결과: 모든 열에 결측치가 있으므로 빈 DataFrame이 됨

df.dropna(axis=1, how="any")


""
0
1
2
3


---

## 결측치 처리 요약

### 1. 결측치 종류
- `np.nan`: 실수(float64) 타입
- `pd.NA`: nullable 타입(Int64, string, boolean)
- `pd.NaT`: 날짜/시간(datetime) 타입

### 2. 결측치 확인
- `df.isnull()`: 결측치 여부 확인
- `df.notnull()`: 결측치가 아닌 값 확인
- `df.isnull().sum()`: 컬럼별 결측치 개수

### 3. 결측치 제거
- `df.dropna()`: 결측치가 있는 행 제거
- `df.dropna(how='all')`: 모두 결측치인 행만 제거
- `df.dropna(thresh=n)`: 유효 값이 n개 미만인 행 제거
- `df.dropna(subset=['A'])`: 특정 컬럼 기준 제거

### 4. 결측치 채우기
- `df.fillna(0)`: 0으로 채우기
- `df.fillna(method='ffill')`: 앞의 값으로 채우기
- `df.fillna(method='bfill')`: 뒤의 값으로 채우기
- `df.fillna(df.mean())`: 평균값으로 채우기

In [43]:
mean_by_subject = df.mean(axis=0)
print(mean_by_subject)

a    2.5
b    2.5
c    4.0
dtype: float64


In [44]:
mean_by_student = df.mean(axis=1)
print(mean_by_student)

0    2.0
1    NaN
2    3.5
3    4.5
dtype: float64


In [45]:
# min, max
max_by_subject = df.max(axis=0)